In [0]:
%pip install --upgrade databricks-sdk
%restart_python

In [0]:
from src.lakebase_utils.lakebase_connect import LakebaseAutoscalingClient

client = LakebaseAutoscalingClient(
    auth_mode="user_oauth",
    connection_string="postgresql://tanveer.singh%40databricks.com@ep-crimson-leaf-e1dz6s0k.database.eastus2.azuredatabricks.net/databricks_postgres?sslmode=require",
    endpoint_path="projects/ts42-demo/branches/br-little-mouse-e19k0naw/endpoints/primary",
)

df = client.select(
    """
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name
    """,
    spark=spark,
)
df.display()

In [0]:
from lakebase_utils.lakebase_api import LakebaseDataApiClient
import requests


from databricks.sdk import WorkspaceClient                                                

with LakebaseDataApiClient(
        base_url="https://ep-crimson-leaf-e1dz6s0k.database.eastus2.azuredatabricks.net/api/2.0/workspace/984752964297111/rest/databricks_postgres",
        auth_mode = "user_oauth", 
        endpoint_path="projects/ts42-demo/branches/br-little-mouse-e19k0naw/endpoints/primary"
    ) as client:
        print(f"base={client.base_url}")

        try:
            # # Single page
            rows = client.get("manual_tests", "synced_cdf_source_table", params={"limit": 5})
            print(f"single-page fetch: {len(rows)} row(s)")
            for row in rows:
                print(" ", row)

            # Paginated iteration (small page size to prove multi-page behavior)
            total = 0
            for row in client.paginate("manual_tests", "synced_cdf_source_table", page_size=1, max_rows=1):
                total += 1
                print(f"paginate(page_size=2, max_rows=1): {total} row(s) yielded")
        except requests.HTTPError as e:
            print(f"HTTP {e.response.status_code}: {e.response.text}")
            raise


